# ND2 Unified Discovery Notebook: Fase 4 - Log-Space Discovery (Octupolo l=3)

Objetivo: Superar el estancamiento del 86% mediante la linealización del problema.
Target: ln|V|
Variables: theta, ln_r

In [ ]:
# 0. Sincronización Robusta (Hard Reset)
import os
is_colab = os.path.exists('/content')
base_path = '/content' if is_colab else '/kaggle/working'
target_dir = os.path.join(base_path, 'ND2')
if not os.path.exists(target_dir): os.system(f"git clone https://github.com/manramirezpi/ND2.git {target_dir}")
os.chdir(target_dir)
os.system("git fetch origin")
os.system("git reset --hard origin/main")
print(f"\n[OK] Entorno listo en: {os.getcwd()}")

In [ ]:
# 1. Setup
!pip install gplearn setproctitle geopandas
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results
if not os.path.exists('weights/checkpoint.pth'):
    !wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth

In [ ]:
# 2. GENERAR DATASET OCTUPOLO LOGARÍTMICO
import numpy as np, json, os
def generate_log_octupole_data(num_samples=2000):
    r = np.random.uniform(0.5, 5.0, num_samples)
    theta = np.random.uniform(0.1, np.pi-0.1, num_samples) # Evitar ceros perfectos para el log
    
    cos_t = np.cos(theta)
    P3 = 0.5 * (5 * cos_t**3 - 3 * cos_t)
    V = P3 / (r**4)
    
    # Transformación Logarítmica
    # Usamos valor absoluto para el log, el AI descubrirá la forma de ln|P3| - 4*ln(r)
    ln_V = np.log(np.abs(V))
    ln_r = np.log(r)
    
    data = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
            "theta": [[0.0, float(t)] for t in theta],
            "ln_r": [[0.0, float(lr)] for lr in ln_r],
            "target": [[0.0, float(lv)] for lv in ln_V]}
    
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(data, open("Multipolos/data/nd2_format/OCTUPOLE_LOG_nd2.json", "w"))
    print("Dataset Octupolo LOGARÍTMICO generado.")

generate_log_octupole_data()

In [ ]:
# 3. BÚSQUEDA EN ESPACIO LOGARÍTMICO
!python3 search.py \
    --data Multipolos/data/nd2_format/OCTUPOLE_LOG_nd2.json \
    --vars theta ln_r \
    --target_var target \
    --episodes 3000 \
    --beam_size 20 \
    --device cuda

In [ ]:
# 4. AUTO-PUSH
def get_token_reloaded():
    try: from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except: pass
    try: from google.colab import userdata; return userdata.get('GITHUB_TOKEN')
    except: pass
    return None
token = get_token_reloaded()
if token:
    os.system("git config --global user.email 'manuel@researcher.phys'")
    os.system("git config --global user.name 'ND2-Auto-Agent'")
    os.system(f"git remote set-url origin https://{token}@github.com/manramirezpi/ND2.git")
    !git add Multipolos/results/ Multipolos/data/nd2_format/OCTUPOLE_LOG_nd2.json
    !git commit -m "Fase 4: Prueba Log-Discovery Octupolo (Prueba de Linealización)"
    !git push origin main
    print("\n[OK] Resultados en GitHub. Avísame.")
else: print("\n[!] GITHUB_TOKEN no encontrado.")